# 🔧 Data Preprocessing
**ClimateTrend Analyzer** | Notebook 2 of 3

---
### Goal
In this notebook we will:
- Handle missing values
- Fix data types
- Engineer new features (Decade, Rolling Mean, Anomaly Flag)
- Save the cleaned dataset for analysis
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='darkgrid')
print('Ready ✅')

## Step 1: Load Raw Data

In [ ]:
df = pd.read_csv('../dataset/raw_data/climate_data.csv')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

## Step 2: Handle Missing Values

**Strategy:**
- Numeric columns → fill with column **mean** (avoids disrupting trends)
- Text columns → fill with `'Unknown'`

In [ ]:
print('Missing values BEFORE:')
print(df.isnull().sum())

# Fill numeric columns with mean
for col in df.select_dtypes(include=[np.number]).columns:
    df[col].fillna(df[col].mean(), inplace=True)

# Fill text columns with 'Unknown'
for col in df.select_dtypes(include=['object']).columns:
    df[col].fillna('Unknown', inplace=True)

print('\nMissing values AFTER:')
print(df.isnull().sum())

## Step 3: Fix Data Types

In [ ]:
# Ensure Year is integer, others are float
if 'Year' in df.columns:
    df['Year'] = df['Year'].astype(int)

for col in ['Temperature_Anomaly', 'CO2_ppm', 'Rainfall_mm']:
    if col in df.columns:
        df[col] = df[col].astype(float)

df.dtypes

## Step 4: Feature Engineering

We create 3 new columns:

| Column | Description |
|--------|-------------|
| `Decade` | 10-year group (e.g., 1980, 1990, 2000) |
| `Temp_RollingMean` | 5-year moving average of temperature |
| `Anomaly_Flag` | 1 if temp anomaly > 0.5°C (extreme event), else 0 |

In [ ]:
# ── Decade ──
if 'Year' in df.columns:
    df['Decade'] = (df['Year'] // 10) * 10

# ── Rolling Mean (sort first!) ──
if 'Temperature_Anomaly' in df.columns:
    df = df.sort_values('Year').reset_index(drop=True)
    df['Temp_RollingMean'] = df['Temperature_Anomaly'].rolling(window=5, min_periods=1).mean()

    # ── Anomaly Flag ──
    df['Anomaly_Flag'] = (df['Temperature_Anomaly'] > 0.5).astype(int)

print('New columns added:')
print(df[['Year', 'Temperature_Anomaly', 'Decade', 'Temp_RollingMean', 'Anomaly_Flag']].head(10))

## Step 5: Verify Cleaned Data

In [ ]:
print(f'Shape: {df.shape}')
df.describe().round(3)

## Step 6: Before vs After – Temperature Comparison

In [ ]:
if 'Temperature_Anomaly' in df.columns:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(df['Year'], df['Temperature_Anomaly'],
            alpha=0.5, color='#78c1f3', label='Raw Anomaly')
    ax.plot(df['Year'], df['Temp_RollingMean'],
            color='#e05c5c', linewidth=2.5, label='5-Year Rolling Mean')
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_title('Temperature Anomaly: Raw vs Smoothed', fontsize=13, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Anomaly (°C)')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Step 7: Save Processed Data

In [ ]:
os.makedirs('../dataset/processed_data', exist_ok=True)
df.to_csv('../dataset/processed_data/climate_cleaned.csv', index=False)
print('✅ Saved to: dataset/processed_data/climate_cleaned.csv')
print(f'   Final shape: {df.shape}')

---
## ✅ Preprocessing Summary

- ✔ Missing values handled
- ✔ Data types corrected
- ✔ New features: Decade, Temp_RollingMean, Anomaly_Flag
- ✔ Dataset saved

➡️ Move on to **visualization.ipynb**